In [ ]:
%matplotlib inline
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATASET = "Dataset/"

Data Understanding and Preparation

In [ ]:
prsa = pd.read_csv(DATASET + "PRSA_data_2010.1.1-2014.12.31.csv")

In [ ]:
prsa.head()

In [ ]:
prsa.info()

In [ ]:
for i in prsa.columns:
    print(f"column: {i}")
    print(prsa[i].unique())
    print()

In [ ]:
# pm2.5 is the only feature with missing values
# Let us replace its missing values with the median
prsa["pm2.5"] = prsa["pm2.5"].fillna(prsa["pm2.5"].median())

In [ ]:
prsa.describe()

In [ ]:
# We create two datasets, one for classification and one for forcasting
prsa_clf = prsa.copy()
prsa_forcast = prsa.copy()

In [ ]:
# Let us decompose the PM2.5 feature into 4 air-quality levels (Good/Moderate/Unhealthy/Hazardous).
# The breakpoints are defined by the US EPA (Environmental Protection Agency) and are as follows:
# 0-9: Good
# 9-35: Moderate
# 35-225: Unhealthy
# 225+: Hazardous

def air_quality_level(pm25):
    if pm25 <= 9:
        return "Good"
    elif pm25 <= 35:
        return "Moderate"
    elif pm25 <= 225:
        return "Unhealthy"
    else:
        return "Hazardous"

prsa_clf["Air Quality"] = prsa_clf["pm2.5"].apply(air_quality_level)

# Now we can drop the PM2.5 column, since we have already created a new column with the air quality levels.
prsa_clf.drop(columns=["pm2.5"], inplace=True)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

features = [feature for feature in prsa_clf.columns if feature != 'Air Quality']

n_cols = 3
n_rows = math.ceil(len(features) / n_cols)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 5*n_rows), constrained_layout=True)

palette = {
    "Good": "green",
    "Moderate": "yellow",
    "Unhealthy": "red",
    "Hazardous": "purple"
   }

hue_order = ["Hazardous", "Unhealthy", "Moderate", "Good"]

for idx, (ax, feature) in enumerate(zip(axes.flat, features)):
    sns.histplot(data=prsa_clf, x=feature, hue='Air Quality',multiple="stack", kde=True, bins=30, ax=ax, hue_order=hue_order, palette=palette)
    ax.set_title(f'Distribution of {feature} by Air Quality')
    if idx >= len(features) - 3:  # just for the last 3 features to make the x-axis more readable
        mean = prsa_clf[feature].mean()
        std = prsa_clf[feature].std()
        ax.set_xlim(mean - 0.25 * std, mean + 1.5*std)

for ax in axes.flat[len(features):]:
    ax.axis('off')

plt.show()

In [ ]:
# Convert the 'cbwd' column to integer values
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
prsa_clf['cbwd'] = label_encoder.fit_transform(prsa_clf['cbwd'])

In [ ]:
# display the correlation matrix
sns.heatmap(prsa.corr(numeric_only=True), vmin=-1, vmax=1, cmap='coolwarm')

In [ ]:
# correlation between year and No is trivial
# A group of 3 features (DEWP, TEMP, PRES) are highly correlated with each other because of physical laws

In [ ]:
# PCA is not necessary for this dataset because the features space is quite small.

In [ ]:
# Let us normalize the data, it's important for the algorithms we will use later.
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
prsa_clf_scaled = scaler.fit_transform(prsa_clf.drop(columns=["Air Quality"]))  # we don't want to scale the target variable

In [ ]:
# Dataset visualization
features = [feature for feature in prsa_forcast.columns if feature != 'No']

plt.figure(figsize=(15, 5*len(features)))
for i, feature in enumerate(features):
    plt.subplot(len(features), 1, i + 1)
    plt.plot(prsa_forcast["No"], prsa_forcast[feature], label=feature)
    plt.title(f"Time series of {feature}")
    plt.xlabel("Time")
    plt.ylabel(feature)
plt.show()

In [ ]:
import torch
import torch.nn as nn

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

Classification

In [ ]:
# We need to encode the feature Air Quality into numerical values
label_encoder = LabelEncoder()
prsa_clf['Air Quality'] = label_encoder.fit_transform(prsa_clf['Air Quality'])

In [ ]:
def split_multivariate_sequence(X, y, window_size):
    XX = np.array([X[i:i+window_size] for i in range(len(X)-window_size)])
    YY = y[window_size:]
    return XX, YY

In [ ]:
# Let us separate the dataset into training and testing sets
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(prsa_clf_scaled, prsa_clf["Air Quality"], test_size=0.3, random_state=42, stratify=prsa_clf["Air Quality"])

window_size = 100
X_train, y_train = split_multivariate_sequence(X_train, y_train, window_size)
X_test, y_test = split_multivariate_sequence(X_test, y_test, window_size)

In [ ]:
# First we try to use a vanilla RNN
from torch.utils.data import DataLoader

train_data = list(zip(X_train, y_train))
test_data = list(zip(X_test, y_test))

batch_size = 32

train_dataloader = DataLoader(train_data, batch_size=batch_size, shuffle=False)
test_dataloader = DataLoader(test_data, batch_size=batch_size, shuffle=False)

print(f"Length of train dataloader: {len(train_dataloader)} batches of {batch_size}")
print(f"Length of test dataloader: {len(test_dataloader)} batches of {batch_size}")

In [ ]:
class VRNN(nn.Module):
    def __init__(self, input_size, hidden_size, num_layer, num_classes):
        super(VRNN, self).__init__()
        self.hidden_size = hidden_size
        self.num_layer = num_layer
        self.rnn = nn.RNN(input_size, hidden_size, num_layer, batch_first=True)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        h0 = torch.zeros(self.num_layer, x.size(0), self.hidden_size).to(device)
        out, _ = self.rnn(x, h0)
        out = self.fc(out[:, -1, :])
        return out

In [ ]:
if False:
    print("deactivated")
else:
    input_size = X_train.shape[2]  # number of features per time step
    num_classes = 4

    # Model hyperparameters
    hidden_size = 32
    num_layer = 1
    num_epochs = 100
    lr = 0.0001

    # Create the model
    model = VRNN(input_size, hidden_size, num_layer, num_classes).to(device)
    loss_function = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    # clipnorm to avoid exploding gradients
    clip = 1.0

    # training loop
    for epoch in range(num_epochs):
        for i, (inputs, labels) in enumerate(train_dataloader):
            inputs = inputs.to(device).float()
            labels = labels.to(device)

            # Forward pass
            outputs = model(inputs)
            loss = loss_function(outputs, labels)

            # Backward pass and optimization
            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), clip)
            optimizer.step()

        print(f'Epoch [{epoch + 1}/{num_epochs}], Loss: {loss.item():.4f}')

In [ ]:
# Problem, the learning seems to diverge... (change hyperparameters, or use a different model)
# Let us try with LSTM and with TensorFlow
import tensorflow as tf

In [ ]:
lstm = tf.keras.models.Sequential([
    tf.keras.layers.LSTM(units=32),
    tf.keras.layers.Dense(units=4, activation="softmax")
])
lstm.build(input_shape=(None, window_size, X_train.shape[2]))  # (batch_size, time_steps, features)
lstm.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
lstm.summary()

In [ ]:
#hist = lstm.fit(X_train, y_train, validation_split=.3, epochs=200)

In [ ]:
if True:
    print("deactivated")
else:
    history = hist.history
    fig, axs = plt.subplots(1, 2, figsize=(12, 5))

    # Loss
    axs[0].plot(history['loss'], label='Train Loss')
    axs[0].plot(history['val_loss'], label='Val Loss')
    axs[0].set_xlabel('Epoch')
    axs[0].set_ylabel('Loss')
    axs[0].legend()

    # Accuracy
    axs[1].plot(history['accuracy'], label='Train Accuracy')
    axs[1].plot(history['val_accuracy'], label='Val Accuracy')
    axs[1].set_xlabel('Epoch')
    axs[1].set_ylabel('Accuracy')
    axs[1].legend()

    plt.tight_layout()
    plt.show()

    lstm.evaluate(X_test, y_test)

In [ ]:
# GRU Model

In [ ]:
gru = tf.keras.models.Sequential([
    tf.keras.layers.GRU(32, kernel_initializer=tf.keras.initializers.Identity(gain=1.0)),
    tf.keras.layers.Dense(units=4, activation="softmax")
])
gru.build(input_shape=(None, window_size, X_train.shape[2]))  # (batch_size, time_steps, features)
gru.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
gru.summary()

In [ ]:
#hist = gru.fit(X_train, y_train, validation_split=.3, epochs=500)

In [ ]:
if True:
    print("deactivated")
else:
    history = hist.history
    fig, axs = plt.subplots(1, 2, figsize=(12, 5))

    # Loss
    axs[0].plot(history['loss'], label='Train Loss')
    axs[0].plot(history['val_loss'], label='Val Loss')
    axs[0].set_xlabel('Epoch')
    axs[0].set_ylabel('Loss')
    axs[0].legend()

    # Accuracy
    axs[1].plot(history['accuracy'], label='Train Accuracy')
    axs[1].plot(history['val_accuracy'], label='Val Accuracy')
    axs[1].set_xlabel('Epoch')
    axs[1].set_ylabel('Accuracy')
    axs[1].legend()

    plt.tight_layout()
    plt.show()

    gru.evaluate(X_test, y_test)

In [ ]:
# 100 epochs is enough, we can see that the model is not overfitting.
# We don't see any improvement with the GRU.

Forecasting

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(prsa_forcast["No"], prsa_forcast["pm2.5"], label="PM2.5")
plt.title("PM2.5 time series")

In [ ]:
# Encode the 'cbwd' column to integer values
label_encoder = LabelEncoder()
prsa_forcast['cbwd'] = label_encoder.fit_transform(prsa_forcast['cbwd'])

# Normalize the data
scaler = StandardScaler()
prsa_forcast_scaled_array = scaler.fit_transform(prsa_forcast)  # ndarray
prsa_forcast_scaled = pd.DataFrame(prsa_forcast_scaled_array, columns=prsa_forcast.columns)  # reconvert to DataFrame

features = [col for col in prsa_forcast_scaled.columns if (col != "No")]
X_full = prsa_forcast_scaled[features].values
y_full = prsa_forcast_scaled["pm2.5"].values

window_size = 100  # hyperparameter, can be tuned

X, y = split_multivariate_sequence(X_full, y_full, window_size)

In [ ]:
# Split the data into train and test sets
split_idx = int(len(X) * 0.7)
X_train = X[:split_idx]
X_test = X[split_idx:]
y_train = y[:split_idx]
y_test = y[split_idx:]

plt.figure(figsize=(20,5))
plt.plot(np.array(prsa_forcast_scaled["pm2.5"]), label="PM2.5")
plt.title("PM2.5 time series")
plt.axvline(split_idx, c="red")
plt.show()

In [ ]:
# Reset the graph and free memory
tf.keras.backend.clear_session()

model = tf.keras.models.Sequential()
model.add(tf.keras.Input(shape=(window_size, len(features))))
model.add(tf.keras.layers.LSTM(64, activation='relu', return_sequences=False))
model.add(tf.keras.layers.Dense(1))
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001, clipnorm=1.0), loss='mse')
model.summary()

In [ ]:
# Deep Encoder-Decoder RNN
from tensorflow.keras import regularizers
# Reset the graph and free memory
tf.keras.backend.clear_session()

model = tf.keras.models.Sequential(
    [tf.keras.layers.LSTM(32, kernel_regularizer=regularizers.l2(1e-4), kernel_initializer='glorot_uniform', return_sequences=True),
    tf.keras.layers.GRU(32, kernel_regularizer=regularizers.l2(1e-4), kernel_initializer='glorot_uniform', return_sequences=False),
    tf.keras.layers.Dense(1, activation="linear")]
    )
model.build(input_shape=(None, window_size, len(features)))
model.compile(
    loss='mean_squared_error',
    optimizer="adam"
    )
model.summary()

In [ ]:
#hist = model.fit(X_train, y_train, epochs=30, validation_split=.3, batch_size=128)

In [ ]:
plt.figure(figsize=(12, 9))
plt.plot(hist.history["loss"][0:], label="train")
plt.plot(hist.history["val_loss"][0:], label="val")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.show()

In [ ]:
model.evaluate(X_test, y_test)

In [ ]:
y_pred = model.predict(X)
idx = np.arange(len(prsa_forcast["No"].values[window_size:]))
plt.figure(figsize=(20,5))
plt.plot(idx, prsa_forcast_scaled["pm2.5"].values[window_size:], c="gray", label="true")
plt.plot(idx[:split_idx], y_pred.ravel()[:split_idx], c="b", label="train")
plt.plot(idx[split_idx:], y_pred.ravel()[split_idx:], c="r", label="test")
plt.legend()
plt.title("PM2.5 time series")
plt.show()